# Facial Emotion Recognition — Kaggle Training Notebook

Trains a CNN across **7 emotion classes**.

**Dataset:** nishchalchandel/raf-db-face-emotion-dataset

**Dataset structure:**
```
train/angry/, train/happy/, ... (7 emotion folders)
validation/angry/, validation/happy/, ...
test/angry/, test/happy/, ...
```

**Active config (best proven setup):**
- Grayscale images at **48×48**
- 3-block custom CNN
- Data augmentation
- EarlyStopping
- ReduceLROnPlateau

**Output:** `facialemotionmodel.h5` saved to `/kaggle/working/`

### Before running:
1. Settings (right panel) → **Accelerator** → **GPU T4 x2**
2. Add dataset: search **nishchalchandel/raf-db-face-emotion-dataset** in the Data tab
3. Click **Save Version** → **Save & Run All (Commit)** to train offline

## Step 1 — Configuration

In [ ]:
import os

# ── Dataset path — from /kaggle/input/datasets/nishchalchandel/raf-db-face-emotion-dataset
# Kaggle mounts external datasets at /kaggle/input/<dataset-slug>
# The slug for this dataset is the last part of the path
DATASET_NAME = 'raf-db-face-emotion-dataset'
INPUT_DIR    = f'/kaggle/input/{DATASET_NAME}'

# Image settings
IMG_SIZE   = 48
CHANNELS   = 1
COLOR_MODE = 'grayscale'
BATCH_SIZE = 64

OUTPUT_DIR = '/kaggle/working'

# ── Print the full directory tree so we can confirm the structure ─────────────
print('Full structure of', INPUT_DIR)
for root, dirs, files in os.walk(INPUT_DIR):
    depth = root.replace(INPUT_DIR, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth >= 2:
        dirs[:] = []

# ── Auto-detect train/, validation/, test/ ───────────────────────────────────
TRAIN_DIR = None
VAL_DIR   = None
TEST_DIR  = None

for root, dirs, files in os.walk(INPUT_DIR):
    basename = os.path.basename(root).lower()
    if basename == 'train'                  and TRAIN_DIR is None: TRAIN_DIR = root
    if basename in ('validation', 'val')    and VAL_DIR   is None: VAL_DIR   = root
    if basename == 'test'                   and TEST_DIR  is None: TEST_DIR  = root
    if TRAIN_DIR and VAL_DIR and TEST_DIR:
        break

# If no validation folder found, fall back to using test as validation
if VAL_DIR is None:
    print('⚠️  No validation/ folder found — using test/ as validation')
    VAL_DIR = TEST_DIR

assert TRAIN_DIR is not None, 'Could not find train/ folder — check dataset structure above'
assert TEST_DIR  is not None, 'Could not find test/ folder — check dataset structure above'

print('\n── Resolved paths ───────────────────')
print('Train      dir:', TRAIN_DIR)
print('Validation dir:', VAL_DIR)
print('Test       dir:', TEST_DIR)
print('Output     dir:', OUTPUT_DIR)
print('\nTrain subfolders     :', sorted(os.listdir(TRAIN_DIR)))
print('Validation subfolders:', sorted(os.listdir(VAL_DIR)))
print('Test subfolders      :', sorted(os.listdir(TEST_DIR)))

## Step 2 — Check GPU

In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)

if gpus:
    print('✅ GPU detected — training will be fast!')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('⚠️  No GPU detected. Go to Settings → Accelerator → GPU T4 x2')

## Step 3 — Imports

In [ ]:
import glob
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import load_img, ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Dense, Conv2D, Dropout, Flatten,
    MaxPooling2D, BatchNormalization
)
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('All imports successful.')

## Step 4 — Build Dataset Dataframes

In [ ]:
def build_dataframe(img_dir):
    image_paths, labels = [], []
    for emotion in sorted(os.listdir(img_dir)):
        emotion_dir = os.path.join(img_dir, emotion)
        if not os.path.isdir(emotion_dir):
            continue
        count = 0
        for imagename in os.listdir(emotion_dir):
            if imagename.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(emotion_dir, imagename))
                labels.append(emotion.lower())
                count += 1
        print(f'  {emotion:<10}: {count} images')
    return image_paths, labels

print('Building train dataframe...')
train_images, train_labels = build_dataframe(TRAIN_DIR)
train_df = pd.DataFrame({'image': train_images, 'label': train_labels})

print('\nBuilding validation dataframe...')
val_images, val_labels = build_dataframe(VAL_DIR)
val_df = pd.DataFrame({'image': val_images, 'label': val_labels})

print('\nBuilding test dataframe...')
test_images, test_labels = build_dataframe(TEST_DIR)
test_df = pd.DataFrame({'image': test_images, 'label': test_labels})

print('\n── Summary ──────────────────────────')
print('Train      samples:', len(train_df))
print('Validation samples:', len(val_df))
print('Test       samples:', len(test_df))
print('Classes:', sorted(train_df['label'].unique()))

## Step 5 — Extract Features & Normalise

In [ ]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image, color_mode=COLOR_MODE, target_size=(IMG_SIZE, IMG_SIZE))
        features.append(np.array(img))
    return np.array(features).reshape(-1, IMG_SIZE, IMG_SIZE, CHANNELS)

print('Extracting train features...')
x_train = extract_features(train_df['image']) / 255.0

print('Extracting validation features...')
x_val = extract_features(val_df['image']) / 255.0

print('Extracting test features...')
x_test = extract_features(test_df['image']) / 255.0

print('x_train shape:', x_train.shape)
print('x_val   shape:', x_val.shape)
print('x_test  shape:', x_test.shape)

## Step 6 — Encode Labels

In [ ]:
le = LabelEncoder()
le.fit(train_df['label'])

y_train = to_categorical(le.transform(train_df['label']), num_classes=7)
y_val   = to_categorical(le.transform(val_df['label']),   num_classes=7)
y_test  = to_categorical(le.transform(test_df['label']),  num_classes=7)

print('Label classes (index order used by model):', list(le.classes_))
print('y_train shape:', y_train.shape)

with open(os.path.join(OUTPUT_DIR, 'label_classes.json'), 'w') as f:
    json.dump(list(le.classes_), f)
print('Label classes saved to label_classes.json')

## Step 7 — Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)

datagen.fit(x_train)
print('Data augmentation configured.')

## Step 8 — Build or Resume Model

In [ ]:
checkpoint_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_files = glob.glob(os.path.join(checkpoint_dir, '*.h5'))
start_epoch = 0

if checkpoint_files:
    latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
    print('Resuming from:', latest_checkpoint)
    model = load_model(latest_checkpoint, compile=False)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    match = re.search(r'facialemotionmodel(\d+)_', os.path.basename(latest_checkpoint))
    if match:
        start_epoch = int(match.group(1))
    print('Resuming from epoch:', start_epoch)

else:
    print('No checkpoint found — building model from scratch')
    model = Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),

        # Block 1
        Conv2D(64, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),   # → 24×24
        Dropout(0.25),

        # Block 2
        Conv2D(128, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),   # → 12×12
        Dropout(0.25),

        # Block 3
        Conv2D(256, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),   # → 6×6
        Dropout(0.25),

        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(7, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

model.summary()

## Step 9 — Train

In [ ]:
checkpoint_cb = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, 'facialemotionmodel{epoch:02d}_valacc{val_accuracy:.4f}.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stop_cb = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    datagen.flow(x_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=len(x_train) // BATCH_SIZE,
    epochs=100,
    initial_epoch=start_epoch,
    validation_data=(x_val, y_val),
    callbacks=[checkpoint_cb, reduce_lr_cb, early_stop_cb]
)

## Step 10 — Save Final Model

In [ ]:
model_json = model.to_json()
with open(os.path.join(OUTPUT_DIR, 'facialemotionmodel.json'), 'w') as f:
    f.write(model_json)

model.save_weights(os.path.join(OUTPUT_DIR, 'facialemotionmodel.weights.h5'))
model.save(os.path.join(OUTPUT_DIR, 'facialemotionmodel.h5'))

print('✅ Files saved to /kaggle/working/:')
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'  {f}  ({size_mb:.1f} MB)')

## Step 11 — Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)
print(f'\n── Final Test Results ───────────────')
print(f'Test accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test loss     : {test_loss:.4f}')

## Step 12 — Plot Training History

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'))
plt.show()
print('Training complete!')